# 중앙대 의료AI 해커톤 — Colab Master Notebook

**Stage 1**: PathMNIST로 워밍업 (파이프라인 검증)
**Stage 2**: 본 게임 데이터 도착 후 dataset.py만 교체

## 실행 순서
1. 환경 셋업 + GitHub 클론
2. PyTorch 2.2.0 다운그레이드 (필요 시)
3. PathMNIST 다운로드
4. View A 학습 → View C 학습
5. Merge grid search
6. Few-shot fine-tuning
7. Inference + submission.csv 생성

## 0. 환경 검증

In [ ]:
import sys, torch
print(f'Python: {sys.version_info[:3]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. PyTorch 2.2.0 다운그레이드 (이미 했으면 SKIP)

위 셀에서 PyTorch가 2.2.x가 아니면 실행. 끝나면 **런타임 → 세션 다시 시작**.

In [ ]:
# 필요할 때만 실행
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121

## 2. GitHub 리포 클론 + 워밍업 패키지 설치

아래 `<USER>/<REPO>`를 본인 GitHub 경로로 수정.

In [ ]:
# %cd /content
# !rm -rf medical_ai_hackathon
# !git clone https://github.com/<USER>/medical_ai_hackathon.git
# %cd medical_ai_hackathon
# !pip install -q -r requirements_warmup.txt
import os, sys
PROJECT_DIR = '/content/medical_ai_hackathon'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
    print('PWD:', os.getcwd())
    print('Files:', os.listdir())

## 3. Google Drive 마운트 (체크포인트 영구 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/medical_ai_ckpts'
os.makedirs(DRIVE_DIR, exist_ok=True)
# checkpoints/ 폴더를 Drive와 symlink (학습 결과 자동 저장)
import os
if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_DIR, 'checkpoints')
print('Checkpoint dir:', os.path.realpath('checkpoints'))

## 4. PathMNIST 다운로드 sanity check

In [ ]:
%cd /content/medical_ai_hackathon/src
import sys; sys.path.insert(0, '.')
from dataset_pathmnist import _load_pathmnist, create_view_datasets, create_fewshot_datasets
cache = _load_pathmnist()
for split in ['train', 'val', 'test']:
    imgs, labels = cache[split]
    print(f'  {split}: imgs={imgs.shape}, labels unique={len(set(labels.tolist()))}')

## 5. Phase 1 — View A 학습

epochs를 줄여서 빠르게 한 번 돌려보고, 잘 되면 늘리세요. 첫 실행은 epochs=20 정도 추천.

In [ ]:
!python train_view.py --view A --dataset pathmnist --epochs 20 --seed 42 --save_init

## 6. Phase 1 — View C 학습

In [ ]:
!python train_view.py --view C --dataset pathmnist --epochs 20 --seed 42

## 7. Phase 2 — Merge Grid Search

4가지 merge 기법 + 여러 λ를 비교. Sagittal validation F1으로 best 선정 → base_model.pth 저장.

In [ ]:
!python merge_and_eval.py --dataset pathmnist --seed 42

## 8. Phase 3 — Few-shot Fine-tuning on Sagittal

In [ ]:
!python train_fewshot.py --dataset pathmnist --seed 42 --epochs 30

## 9. Inference (sanity check)

PathMNIST에는 별도 test 폴더가 없으니, 워밍업에선 이 셀을 실행하지 않아도 됩니다.
본 게임 단계에서 활용.

In [ ]:
# 본 게임 데이터 도착 후:
# !python inference.py --model_path checkpoints/model.pth --test_dir /content/data/test --output submission.csv
# !head submission.csv

## 10. 결과 확인 + 다음 실험

- `checkpoints/merge_grid_results.csv` 에 grid search 결과 정리됨
- 어떤 merge 기법이 가장 좋았는지 확인
- λ_EWC 등 하이퍼파라미터 조정해서 재실행

In [ ]:
import pandas as pd
df = pd.read_csv('checkpoints/merge_grid_results.csv')
print(df.sort_values('sag_f1', ascending=False).to_string(index=False))